# 28 — Drive Write Benchmark
Measures how long it takes to save a ~3 GB array to the new D:\ and E:\ mounted drives.

In [1]:
import time
import os
import numpy as np
from pathlib import Path

## Config

In [2]:
TARGET_GB = 3.0
DRIVES = [Path("D:/"), Path("E:/")]
FILENAME = "benchmark_3gb.npy"
N_REPEATS = 3  # runs per drive to get a stable average

## Allocate data in RAM

In [3]:
n_bytes = int(TARGET_GB * 1024**3)
n_elements = n_bytes // 4  # float32 = 4 bytes
data = np.random.default_rng(42).random(n_elements, dtype=np.float32)
actual_gb = data.nbytes / 1024**3
print(f"Array shape: {data.shape}  |  size: {actual_gb:.2f} GB")

Array shape: (805306368,)  |  size: 3.00 GB


## Benchmark

In [4]:
results = {}  # drive -> list of (write_s, read_s)

for drive in DRIVES:
    dest = drive / FILENAME
    
    if not drive.exists():
        print(f"[SKIP] {drive} not found")
        continue

    free_gb = os.statvfs(drive).f_bavail * os.statvfs(drive).f_frsize / 1024**3 if hasattr(os, 'statvfs') else None
    # On Windows use shutil
    try:
        import shutil
        free_gb = shutil.disk_usage(drive).free / 1024**3
    except Exception:
        pass
    if free_gb is not None:
        print(f"\n{drive}  —  {free_gb:.1f} GB free")
    else:
        print(f"\n{drive}")

    write_times, read_times = [], []

    for rep in range(1, N_REPEATS + 1):
        # --- WRITE ---
        t0 = time.perf_counter()
        np.save(dest, data)
        write_s = time.perf_counter() - t0
        write_times.append(write_s)

        # --- READ BACK ---
        t0 = time.perf_counter()
        _ = np.load(dest)
        read_s = time.perf_counter() - t0
        read_times.append(read_s)

        print(
            f"  rep {rep}/{N_REPEATS}  "
            f"write {write_s:.2f}s ({actual_gb/write_s:.1f} GB/s)  "
            f"read {read_s:.2f}s ({actual_gb/read_s:.1f} GB/s)"
        )

        dest.unlink()  # remove after each rep so we don't fill the drive

    results[str(drive)] = {"write": write_times, "read": read_times}

print("\nDone.")


D:\  —  3725.7 GB free
  rep 1/3  write 1.99s (1.5 GB/s)  read 0.98s (3.1 GB/s)
  rep 2/3  write 1.63s (1.8 GB/s)  read 1.15s (2.6 GB/s)
  rep 3/3  write 2.19s (1.4 GB/s)  read 1.02s (2.9 GB/s)

E:\  —  3725.7 GB free
  rep 1/3  write 2.17s (1.4 GB/s)  read 1.04s (2.9 GB/s)
  rep 2/3  write 2.24s (1.3 GB/s)  read 0.99s (3.0 GB/s)
  rep 3/3  write 2.28s (1.3 GB/s)  read 0.99s (3.0 GB/s)

Done.


## Summary

In [5]:
print(f"{'Drive':<8} {'Avg write (s)':>14} {'Avg write (GB/s)':>17} {'Avg read (s)':>13} {'Avg read (GB/s)':>16}")
print("-" * 72)
for drive, times in results.items():
    avg_w = np.mean(times["write"])
    avg_r = np.mean(times["read"])
    print(
        f"{drive:<8} {avg_w:>14.2f} {actual_gb/avg_w:>17.2f} "
        f"{avg_r:>13.2f} {actual_gb/avg_r:>16.2f}"
    )

Drive     Avg write (s)  Avg write (GB/s)  Avg read (s)  Avg read (GB/s)
------------------------------------------------------------------------
D:\                1.94              1.55          1.05             2.85
E:\                2.23              1.34          1.01             2.98
